<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Notebook 5 ⭐ Premium — ML · Prédiction de sous-performance
### 📝 VERSION APPRENANT

> Objectif : Prédire si une campagne va sous-performer dès J3.
> Contrainte métier : **Recall ≥ 0.75** — mieux vaut sur-alerter que manquer.
> Livrable : `campagnes_risque_scores.csv` avec scores et niveaux d'alerte.

---
## 0. Setup

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine scikit-learn --quiet

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, sys

from sklearn.linear_model   import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing  import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import roc_auc_score, f1_score, recall_score, precision_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve
from sklearn.model_selection import TimeSeriesSplit
import joblib

COLORS = {'primary':'#534AB7','secondary':'#1D9E75','warning':'#EF9F27','danger':'#E24B4A','neutral':'#888780'}
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#F9F9F8','axes.grid':True,'grid.alpha':0.35,'font.size':11})
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
print('Imports ML OK ✅')


---
## 1. Chargement et exploration

### MÉTHODE
Une ligne = une campagne. `campagne_sous_performe` = variable cible (26.3% positifs). Distribution favorable — pas besoin de SMOTE.

In [ ]:
# 📝 TODO : Charger marketpulse_analytics.csv et afficher la distribution de la variable cible
# 💡 Indice 1 : Essayer SAVE_PATH d'abord, sinon GitHub
# 💡 Indice 2 : df['campagne_sous_performe'].value_counts() pour la distribution
# 💡 Indice 3 : Afficher : nb positifs, nb négatifs, %

analytics_path = f'{SAVE_PATH}marketpulse_analytics.csv'
df = pd.read_csv(analytics_path, ...) if os.path.exists(analytics_path) else pd.read_csv(BASE_URL + 'marketpulse_analytics.csv', parse_dates=['date_debut','date_fin'])
print(f'Dimensions : {df.shape}')
vc = df['campagne_sous_performe'].value_counts()
# Afficher distribution


### 🧠 Tes observations

> - Le taux de positifs (~26%) est-il favorable pour le ML ? Pourquoi ce taux ne nécessite-t-il pas de SMOTE ?
> - Combien de campagnes dans le dataset ? Sur quelle période ?


---
## 2. Sélection des features — Exclusion du leakage

### MÉTHODE — Anti-leakage temporel
La prédiction se fait à **J3** — inclure des données de fin de campagne serait du leakage.

**Leakage à exclure :**
| Feature | Raison |
|---|---|
| `roas_total` | Performance finale — inconnue à J3 |
| `ctr_total` | Calculé sur toute la campagne |
| `conversions_total` | Total final — inconnu à J3 |
| `campagne_sous_performe` | C'est la cible ! |

In [ ]:
# 📝 TODO : Définir LEAKAGE_COLS et FEATURES en respectant la contrainte J3
# 💡 Indice 1 : FEATURES = colonnes disponibles à J3 : ctr_j3, roas_j3, taux_conv_j3, variation_ctr_j1_j3, ...
# 💡 Indice 2 : Inclure aussi : taux_sous_perf_canal, taux_sous_perf_client, duree_jours, canal_num...
# 💡 Indice 3 : Vérifier qu'aucune feature de FEATURES n'est dans LEAKAGE_COLS

LEAKAGE_COLS = ['roas_total','ctr_total','cpa_total','taux_conv_total','conversions_total','campagne_sous_performe']
FEATURES = [
    # Signal J3
    'ctr_j3','roas_j3','taux_conv_j3','ctr_j1','roas_j1',
    'variation_ctr_j1_j3','variation_roas_j1_j3','ratio_dep_j3_vs_budget',
    # Historique
    'taux_sous_perf_canal','taux_sous_perf_client','taux_sous_perf_objectif',
    # Campagne
    'duree_jours','est_lancement_weekend','mois_sin','mois_cos',
    'canal_num','objectif_num','type_audience_num','ratio_budget_mensuel',
    # Audience
    'indice_saturation_j3',
]
missing = [f for f in FEATURES if f not in df.columns]
print(f'Features manquantes : {missing}')
print(f'{len(FEATURES)} features retenues')


### 🧠 Tes observations

> - Pourquoi `roas_total` est-il du leakage mais `roas_j3` ne l'est pas ?
> - Si tu utilisais `roas_total` comme feature, tes métriques de test seraient-elles fiables ?


---
## 3. Coupure temporelle stricte

### MÉTHODE
KFold **interdit** sur des données temporelles — entraîner sur le futur pour prédire le passé est un oracle. La coupure temporelle imite les conditions de production.

```
Juin 2022 ─────────── Juin 2023 │ Juil 2023 ──────── Mai 2024
◄──────── TRAIN (~60%) ─────────►│◄──── TEST (~40%) ─────────►
```

In [ ]:
# 📝 TODO : Créer le split temporel à la date de coupure 2023-07-01
# 💡 Indice 1 : DATE_COUPURE = pd.Timestamp('2023-07-01')
# 💡 Indice 2 : mask_train = df_ml['date_debut'] < DATE_COUPURE
# 💡 Indice 3 : mask_test  = df_ml['date_debut'] >= DATE_COUPURE
# 💡 Indice 4 : X_train = df_ml.loc[mask_train, FEATURES]

DATE_COUPURE = pd.Timestamp('2023-07-01')
# Imputation NaN par médiane du canal
df_ml = df[FEATURES + ['campagne_sous_performe','campagne_id','date_debut','canal']].copy().sort_values('date_debut').reset_index(drop=True)
for col in FEATURES:
    if df_ml[col].isnull().any():
        med_canal = df_ml.groupby('canal')[col].transform('median')
        df_ml[col] = df_ml[col].fillna(med_canal).fillna(df_ml[col].median())
mask_train = ...
mask_test  = ...
X_train = ...; X_test = ...; y_train = ...; y_test = ...
print(f'Train : {len(X_train)} | Positifs : {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Test  : {len(X_test)}  | Positifs : {y_test.sum()} ({y_test.mean()*100:.1f}%)')


### 🧠 Tes observations

> - Le taux de positifs est-il similaire dans train et test ? Sinon, cela signifie quoi ?
> - Pourquoi au moins 12 mois d'entraînement sont-ils recommandés ?


---
## 4. Entraînement des 3 modèles

### MÉTHODE
`class_weight='balanced'` donne plus de poids aux positifs (26%) sans sur-échantillonnage. On compare les 3 modèles à seuil par défaut 0.5 avant optimisation.

In [ ]:
# 📝 TODO : Entraîner Logistic Regression + StandardScaler, Random Forest et Gradient Boosting
# 💡 Indice 1 : pipe_lr = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', ...))])
# 💡 Indice 2 : rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', ...)
# 💡 Indice 3 : gb = GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, ...)

# Modèle 1 : Logistic Regression
pipe_lr = Pipeline([('scaler', ...), ('clf', ...)])
pipe_lr.fit(X_train, y_train)
y_prob_lr = ...  # predict_proba(...)[: ,1]
y_pred_lr = ...
# Modèle 2 : Random Forest
rf = ...
rf.fit(X_train, y_train)
y_prob_rf = ...
y_pred_rf = ...
# Modèle 3 : Gradient Boosting
gb = ...
gb.fit(X_train, y_train)
y_prob_gb = ...
y_pred_gb = ...
print('3 modèles entraînés ✅')


In [ ]:
# 📝 TODO : Construire le tableau comparatif AUC / F1 / Recall / Precision à seuil 0.5
# 💡 Indice 1 : from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score
# 💡 Indice 2 : Créer un DataFrame avec une ligne par modèle
# 💡 Indice 3 : Vérifier quelle contrainte Recall >= 0.75 est satisfaite

def metriques(y_true, y_pred, y_prob, nom):
    return {'Modèle':nom,'AUC':round(roc_auc_score(y_true,y_prob),4),'F1':round(f1_score(y_true,y_pred),4),'Recall':round(recall_score(y_true,y_pred),4),'Precision':round(precision_score(y_true,y_pred),4)}
df_comp = pd.DataFrame([
    metriques(y_test, y_pred_lr, y_prob_lr, 'Logistic Regression'),
    metriques(y_test, y_pred_rf, y_prob_rf, 'Random Forest'),
    metriques(y_test, y_pred_gb, y_prob_gb, 'Gradient Boosting'),
]).set_index('Modèle')
print(df_comp)
print(f'Contrainte Recall >= 0.75 satisfaite ?')


### 🧠 Tes observations

> - Quel modèle a le meilleur AUC ? Quel modèle a le meilleur Recall ?
> - Aucun modèle atteint Recall >= 0.75 à seuil 0.5 — pourquoi ? Que faut-il faire ?


---
## 5. Courbes ROC et Precision-Recall

### MÉTHODE
La courbe ROC montre le compromis TPR/FPR. La courbe Precision-Recall est plus informative sur les datasets déséquilibrés. La ligne verticale Recall=0.75 montre la Precision atteignable à notre contrainte.

In [ ]:
# 📝 TODO : Tracer les courbes ROC et Precision-Recall pour les 3 modèles
# 💡 Indice 1 : from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score
# 💡 Indice 2 : fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 💡 Indice 3 : Pour chaque modèle : fpr, tpr, _ = roc_curve(y_test, y_prob)
# 💡 Indice 4 : axvline(0.75) sur le graphique Precision-Recall pour montrer le seuil cible

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for nom, y_prob, color in [('LR', y_prob_lr, COLORS['neutral']), ('RF', y_prob_rf, COLORS['primary']), ('GB', y_prob_gb, COLORS['secondary'])]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{nom} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
# [Courbes Precision-Recall sur axes[1]]
plt.savefig(f'{SAVE_PATH}marketpulse_roc.png', dpi=150); plt.show()


### 🧠 Tes observations

> - Quel modèle a la courbe ROC la plus haute ? Quelle est son AUC ?
> - Sur la courbe Precision-Recall, quelle est la Precision atteignable pour Recall=0.75 avec Random Forest ?


---
## 6. Optimisation du seuil

### MÉTHODE
Boucle sur les seuils 0.25 → 0.70. Le seuil optimal = le plus élevé satisfaisant Recall >= 0.75 (minimiser les fausses alarmes tout en respectant la contrainte).

In [ ]:
# 📝 TODO : Trouver le seuil optimal pour Random Forest : boucle 0.25 → 0.70, contrainte Recall >= 0.75
# 💡 Indice 1 : for seuil in np.arange(0.25, 0.71, 0.05):
# 💡 Indice 2 :     y_pred_s = (y_prob_rf >= seuil).astype(int)
# 💡 Indice 3 :     rec = recall_score(y_test, y_pred_s)
# 💡 Indice 4 :     Afficher : seuil, AUC, F1, Recall, Precision, 'OK' si Recall >= 0.75
# 💡 Indice 5 : SEUIL_OPTIMAL = le plus élevé avec Recall >= 0.75

print(f'{'Seuil':>8} {'AUC':>8} {'Recall':>8} {'Precision':>10}')
resultats = []
for seuil in np.arange(0.25, 0.71, 0.05):
    seuil = round(seuil, 2)
    y_pred_s = (y_prob_rf >= seuil).astype(int)
    if y_pred_s.sum() == 0: continue
    rec = ...; prec = ...; f1 = ...; auc = roc_auc_score(y_test, y_prob_rf)
    ok = '✅' if rec >= 0.75 else '❌'
    print(f'{seuil:>8.2f} {auc:>8.4f} {rec:>8.4f} {prec:>10.4f}  {ok}')
    resultats.append({'seuil':seuil,'recall':rec,'precision':prec,'f1':f1})
df_seuils = pd.DataFrame(resultats)
SEUIL_OPTIMAL = float(df_seuils[df_seuils['recall'] >= 0.75]['seuil'].max())
print(f'\n🎯 Seuil optimal : {SEUIL_OPTIMAL}')
y_pred_final = (y_prob_rf >= SEUIL_OPTIMAL).astype(int)


### 🧠 Tes observations

> - Quel est le seuil optimal ? Quel est le Recall à ce seuil ?
> - Quelle est la Precision à ce seuil ? Combien de fausses alarmes par vrai positif ?


### Visualisation : métriques vs seuil + matrice de confusion

In [ ]:
# 📝 TODO : Tracer la courbe Recall/Precision/F1 vs seuil + matrice de confusion
# 💡 Indice 1 : fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 💡 Indice 2 : axes[0] : ax.plot(seuils, recalls, ...) pour les 3 métriques + axvline seuil optimal
# 💡 Indice 3 : axes[1] : imshow de la matrice de confusion avec annotations TN/FP/FN/TP

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred_final)
# Panel 1 : courbe métriques vs seuil
# Panel 2 : matrice de confusion colorée
# TN vert / FP orange / FN rouge / TP vert
plt.savefig(f'{SAVE_PATH}marketpulse_threshold.png', dpi=150); plt.show()


---
## 7. Validation TimeSeriesSplit (5 folds)

### MÉTHODE
`TimeSeriesSplit(n_splits=5)` crée 5 folds en expansion temporelle — respecte l'ordre temporel. La variance entre folds mesure la stabilité du modèle.

In [ ]:
# 📝 TODO : Valider Random Forest avec TimeSeriesSplit 5 folds au seuil optimal
# 💡 Indice 1 : tscv = TimeSeriesSplit(n_splits=5)
# 💡 Indice 2 : for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_all), 1):
# 💡 Indice 3 :     Entraîner RF sur Xtr, prédire sur Xte au SEUIL_OPTIMAL
# 💡 Indice 4 :     Calculer AUC, F1, Recall, Precision pour ce fold
# 💡 Indice 5 : Afficher moyenne ± écart-type

tscv = TimeSeriesSplit(n_splits=5)
df_ml_sorted = df_ml.sort_values('date_debut').reset_index(drop=True)
X_all = df_ml_sorted[FEATURES]; y_all = df_ml_sorted['campagne_sous_performe']
scores = []
for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_all), 1):
    Xtr, Xte = X_all.iloc[tr_idx], X_all.iloc[te_idx]
    ytr, yte = y_all.iloc[tr_idx], y_all.iloc[te_idx]
    if ytr.sum() < 3 or yte.sum() < 2: continue
    rf_fold = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE)
    rf_fold.fit(Xtr, ytr)
    prob = rf_fold.predict_proba(Xte)[:,1]
    pred = (prob >= SEUIL_OPTIMAL).astype(int)
    scores.append({'auc':roc_auc_score(yte,prob),'recall':recall_score(yte,pred),'f1':f1_score(yte,pred)})
df_tscv = pd.DataFrame(scores)
print('Moyenne ± std :')
print(df_tscv.mean().round(4)); print(df_tscv.std().round(4))


### 🧠 Tes observations

> - L'écart-type d'AUC est-il < 0.05 (modèle stable) ou > 0.10 (instable) ?
> - Les derniers folds (plus de données en train) ont-ils de meilleures performances ?


---
## 8. Feature Importance

### MÉTHODE
Feature importance native du Random Forest (impureté de Gini). Les 5 premières features devraient inclure `roas_j3` et `variation_ctr_j1_j3` — les signaux précoces les plus directs.

In [ ]:
# 📝 TODO : Calculer et visualiser la feature importance du Random Forest
# 💡 Indice 1 : importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
# 💡 Indice 2 : ax.barh(range(len(top10)), top10.values[::-1])
# 💡 Indice 3 : Afficher le cumul des 5 premières features

importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 7))
# Barres horizontales avec couleurs selon l'importance
# ax.barh(...)
plt.savefig(f'{SAVE_PATH}marketpulse_feature_importance.png', dpi=150); plt.show()
print(f'Top 5 features : {importances.index[:5].tolist()}')
print(f'Cumul top 5 : {importances.iloc[:5].sum()*100:.1f}%')


### 🧠 Tes observations

> - Les 5 features les plus importantes sont-elles celles attendues (`roas_j3`, `variation_ctr_j1_j3`) ?
> - La feature `taux_sous_perf_canal` est-elle dans le top 10 ? Que cela valide-t-il ?


---
## 9. Tableau comparatif final — Seuil optimal

### MÉTHODE
Recomparer les 3 modèles au seuil optimal pour la décision finale.

In [ ]:
# 📝 TODO : Tableau comparatif des 3 modèles au seuil optimal + décision modèle retenu
# 💡 Indice 1 : Même structure que la section 4 mais au SEUIL_OPTIMAL
# 💡 Indice 2 : Afficher les métriques + vérifier la contrainte Recall >= 0.75
# 💡 Indice 3 : Sélectionner le modèle à retenir

print(f'=== COMPARAISON FINALE — Seuil {SEUIL_OPTIMAL} ===')
for nom, y_prob in [('LR',y_prob_lr),('RF',y_prob_rf),('GB',y_prob_gb)]:
    y_pred_s = (y_prob >= SEUIL_OPTIMAL).astype(int)
    # Afficher métriques + contrainte OK/KO


---
## 10. Export `campagnes_risque_scores.csv`

### MÉTHODE
3 niveaux : 🔴 Critique (score>0.75) · 🟠 Élevé (0.55-0.75) · 🟡 Modéré (seuil-0.55). Scores OOB (out-of-bag) sur le train pour éviter le leakage.

In [ ]:
# 📝 TODO : Générer les scores pour toutes les campagnes et exporter campagnes_risque_scores.csv
# 💡 Indice 1 : rf_oob = RandomForestClassifier(..., oob_score=True) — entraîner sur X_train
# 💡 Indice 2 : scores_train_oob = rf_oob.oob_decision_function_[:,1]
# 💡 Indice 3 : scores_test = rf.predict_proba(X_test)[:,1]
# 💡 Indice 4 : Définir niveau_alerte : score>0.75→Critique, 0.55→Élevé, seuil→Modéré
# 💡 Indice 5 : Ajouter action_recommandee pour chaque niveau

rf_oob = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, oob_score=True, n_jobs=-1)
rf_oob.fit(X_train, y_train)
scores_train_oob = rf_oob.oob_decision_function_[:,1]
scores_test = rf.predict_proba(X_test)[:,1]
# Assembler scores dans df_ml
scores_all = np.zeros(len(df_ml))
scores_all[mask_train.values] = scores_train_oob
scores_all[mask_test.values] = scores_test
df_ml['score_risque'] = scores_all.round(4)
# Fonction niveau_alerte
def niveau_alerte(score): ...
df_ml['niveau_alerte'] = ...
df_ml['action_recommandee'] = df_ml['niveau_alerte'].map({'Critique':'Pause immédiate','Élevé':'Révision ciblage 48h','Modéré':'Surveillance renforcée','Normal':'Aucune action requise'})
df_scores = df_ml[['campagne_id','canal','date_debut','score_risque','niveau_alerte','action_recommandee','campagne_sous_performe']].sort_values('score_risque',ascending=False)
df_scores.to_csv(f'{SAVE_PATH}campagnes_risque_scores.csv', index=False)
print('Distribution :')
print(df_scores['niveau_alerte'].value_counts())


### 🧠 Tes observations

> - Combien d'alertes Critiques as-tu générées ? Est-ce cohérent avec le nombre de campagnes sous-performantes (93) ?
> - Le fichier est-il correctement trié par score décroissant ? Quel est le score de la campagne en tête ?


---
## 11. Visualisation synthèse

In [ ]:
# 📝 TODO : Créer la visualisation synthèse ML en 5 panels
# 💡 Indice 1 : fig = plt.figure(figsize=(16,12)); gs = gridspec.GridSpec(2,3)
# 💡 Indice 2 : Panel 1 (gs[0,0]) : histogramme scores positifs vs négatifs
# 💡 Indice 3 : Panel 2 (gs[0,1:]) : feature importance top 10
# 💡 Indice 4 : Panel 3 (gs[1,0]) : alertes par canal
# 💡 Indice 5 : Panel 4 (gs[1,1]) : donut niveaux d'alerte
# 💡 Indice 6 : Panel 5 (gs[1,2]) : stabilité TimeSeriesSplit

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)
# Panel 1 : distribution scores
ax1 = fig.add_subplot(gs[0, 0])
# ax1.hist(scores positifs, ...); ax1.hist(scores négatifs, ...)
# Panel 2 : feature importance
ax2 = fig.add_subplot(gs[0, 1:])
# ax2.barh(...)
# [Panels 3,4,5]
plt.savefig(f'{SAVE_PATH}marketpulse_ml_synthese.png', dpi=150, bbox_inches='tight'); plt.show()


---
## 12. Sauvegarde du modèle

In [ ]:
# 📝 TODO : Entraîner le modèle final sur tout le dataset et sauvegarder avec joblib
# 💡 Indice 1 : rf_prod = RandomForestClassifier(...) — entraîner sur TOUTES les données (X_all, y_all)
# 💡 Indice 2 : joblib.dump(rf_prod, f'{SAVE_PATH}marketpulse_rf_model.pkl')
# 💡 Indice 3 : Sauvegarder aussi FEATURES et SEUIL_OPTIMAL

rf_prod = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_prod.fit(df_ml[FEATURES], df_ml['campagne_sous_performe'])
joblib.dump(rf_prod, f'{SAVE_PATH}marketpulse_rf_model.pkl')
joblib.dump(FEATURES, f'{SAVE_PATH}marketpulse_features.pkl')
joblib.dump(SEUIL_OPTIMAL, f'{SAVE_PATH}marketpulse_seuil.pkl')
print('Modèle sauvegardé ✅')


---
## Bilan NB5 — À compléter

### Performances du modèle final (Random Forest)

| Métrique | Valeur | Contrainte |
|---|---|---|
| AUC | ___ | — |
| Recall | ___ | ✅ ≥ 0.75 |
| Precision | ___ | Acceptable |
| F1 | ___ | — |
| Seuil optimal | ___ | — |

### Top 5 features prédictives

1. ✍️
2. ✍️
3. ✍️
4. ✍️
5. ✍️

### Recommandation ré-entraînement

> Recommandes-tu un ré-entraînement mensuel, trimestriel ou annuel ? Justifie.

### Fichiers produits

| Fichier | Créé ? |
|---|---|
| `campagnes_risque_scores.csv` | ☐ |
| `marketpulse_rf_model.pkl` | ☐ |
| `marketpulse_seuil.pkl` | ☐ |


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.